# ManoVarta Colab Runner (No Credentials Needed)

This notebook is designed to run on a friend's Colab independently.

- Uses open models by default (no Hugging Face token needed)
- Does not use GCP / Vertex / your local credentials
- Clones the repository fresh and runs training + evaluation locally in Colab runtime


In [ ]:
import os
import sys
import json
import shutil
import pathlib
import subprocess

# --- User-editable config ---
REPO_URL = "https://github.com/RitwijParmar/ManoVarta.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/multilingualChatbot"

# Open checkpoints: no token required
EXTRACTOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SAFETY_MODEL = "ai4bharat/IndicBERTv2-MLM-only"

# Runtime knobs
DEVICE = "cuda"  # use "cpu" only if no GPU
USE_4BIT = True   # recommended on Colab T4/L4 for extractor
QUICK_RUN = True  # True = faster sanity run, False = deeper run

if QUICK_RUN:
    EXTRACTOR_EPOCHS = 1
    SAFETY_EPOCHS = 1
    EXTRACTOR_EVAL_LIMIT = 48
else:
    EXTRACTOR_EPOCHS = 2
    SAFETY_EPOCHS = 3
    EXTRACTOR_EVAL_LIMIT = 0

OUT_ROOT = pathlib.Path("/content/manovarta_outputs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACTOR_OUT = str(OUT_ROOT / "extractor_qwen25_1_5b")
SAFETY_OUT = str(OUT_ROOT / "safety_indicbert")
REPORTS_DIR = str(OUT_ROOT / "reports")
ARTIFACTS_DIR = str(OUT_ROOT / "artifacts")

print(json.dumps({
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "extractor_model": EXTRACTOR_MODEL,
    "safety_model": SAFETY_MODEL,
    "device": DEVICE,
    "use_4bit": USE_4BIT,
    "quick_run": QUICK_RUN,
    "output_root": str(OUT_ROOT)
}, indent=2))


In [ ]:
# Clone + install
subprocess.run(["nvidia-smi"], check=False)

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR
], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[training]"], check=True)

print("Repo ready at:", REPO_DIR)


In [ ]:
# Build processed datasets (from repo seed/gold assets)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "tools/generate_seed_scaleup.py"], check=True)
subprocess.run([sys.executable, "tools/create_data_splits.py"], check=True)
subprocess.run([
    sys.executable,
    "tools/export_training_sets.py",
    "--extractor-style", "compact"
], check=True)

print("Processed files:")
for path in [
    "data/processed/extractor_train_best.jsonl",
    "data/processed/extractor_dev.jsonl",
    "data/processed/extractor_test.jsonl",
    "data/processed/safety_train.jsonl",
    "data/processed/safety_dev.jsonl",
    "data/processed/safety_test.jsonl",
]:
    print(path, "exists" if pathlib.Path(path).exists() else "missing")


In [ ]:
# Fine-tune extractor (open model, no token)
os.chdir(REPO_DIR)
cmd = [
    sys.executable,
    "-m",
    "training.finetune_extractor",
    "--model-name", EXTRACTOR_MODEL,
    "--train-file", "data/processed/extractor_train_best.jsonl",
    "--eval-file", "data/processed/extractor_dev.jsonl",
    "--output-dir", EXTRACTOR_OUT,
    "--epochs", str(EXTRACTOR_EPOCHS),
    "--batch-size", "1",
    "--grad-accum", "8",
    "--max-length", "1024",
    "--device", DEVICE,
    "--precision", "auto",
    "--gradient-checkpointing",
    "--save-strategy", "steps",
    "--save-steps", "10",
    "--eval-strategy", "no",
    "--resume-from-checkpoint", "last"
]
if USE_4BIT and DEVICE == "cuda":
    cmd.append("--use-4bit")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Extractor output:", EXTRACTOR_OUT)


In [ ]:
# Evaluate extractor checkpoint
os.chdir(REPO_DIR)
extractor_eval_path = pathlib.Path(REPORTS_DIR) / "extractor_eval.json"
extractor_eval_path.parent.mkdir(parents=True, exist_ok=True)

eval_cmd = [
    sys.executable,
    "training/evaluate_extractor_checkpoint.py",
    "--model-path", EXTRACTOR_OUT,
    "--eval-file", "data/processed/extractor_test.jsonl",
    "--device", DEVICE
]
if EXTRACTOR_EVAL_LIMIT > 0:
    eval_cmd.extend(["--limit", str(EXTRACTOR_EVAL_LIMIT)])

print("Running:", " ".join(eval_cmd))
extractor_eval_raw = subprocess.check_output(eval_cmd, text=True)
extractor_eval_path.write_text(extractor_eval_raw, encoding="utf-8")
extractor_eval = json.loads(extractor_eval_raw)
print(json.dumps({
    "example_count": extractor_eval.get("example_count"),
    "overall": extractor_eval.get("overall")
}, indent=2))


In [ ]:
# Train safety classifier
os.chdir(REPO_DIR)
safety_cmd = [
    sys.executable,
    "-m",
    "training.train_safety_classifier",
    "--model-name", SAFETY_MODEL,
    "--train-file", "data/processed/safety_train.jsonl",
    "--eval-file", "data/processed/safety_dev.jsonl",
    "--output-dir", SAFETY_OUT,
    "--epochs", str(SAFETY_EPOCHS),
    "--batch-size", "8",
    "--device", DEVICE,
    "--precision", "auto",
    "--save-strategy", "steps",
    "--save-steps", "10",
    "--eval-strategy", "steps",
    "--eval-steps", "10",
    "--resume-from-checkpoint", "last"
]

print("Running:", " ".join(safety_cmd))
subprocess.run(safety_cmd, check=True)
print("Safety output:", SAFETY_OUT)


In [ ]:
# Evaluate safety checkpoint
os.chdir(REPO_DIR)
safety_eval_path = pathlib.Path(REPORTS_DIR) / "safety_eval.json"
safety_eval_cmd = [
    sys.executable,
    "training/evaluate_safety_checkpoint.py",
    "--model-path", SAFETY_OUT,
    "--eval-file", "data/processed/safety_test.jsonl",
    "--device", DEVICE
]
print("Running:", " ".join(safety_eval_cmd))
safety_eval_raw = subprocess.check_output(safety_eval_cmd, text=True)
safety_eval_path.write_text(safety_eval_raw, encoding="utf-8")
safety_eval = json.loads(safety_eval_raw)
print(json.dumps({
    "accuracy": safety_eval.get("accuracy"),
    "macro_f1": safety_eval.get("macro_f1"),
    "examples": safety_eval.get("examples")
}, indent=2))


In [ ]:
# Final report bundle and zip (still no external credentials)
os.chdir(REPO_DIR)
finalize_cmd = [
    sys.executable,
    "tools/finalize_colab_run.py",
    "--reports-dir", REPORTS_DIR,
    "--outputs-dir", str(OUT_ROOT),
    "--artifacts-dir", ARTIFACTS_DIR,
    "--checkpoint-path", EXTRACTOR_OUT,
    "--safety-checkpoint-path", SAFETY_OUT,
    "--checkpoint-eval-json", str(pathlib.Path(REPORTS_DIR) / "extractor_eval.json"),
    "--safety-eval-json", str(pathlib.Path(REPORTS_DIR) / "safety_eval.json"),
    "--semantic-model", SAFETY_MODEL,
    "--skip-semantic",
    "--skip-llm"
]
print("Running:", " ".join(finalize_cmd))
subprocess.run(finalize_cmd, check=True)

print("\\nKey outputs:")
print("- Extractor:", EXTRACTOR_OUT)
print("- Safety:", SAFETY_OUT)
print("- Reports:", REPORTS_DIR)
print("- Artifacts:", ARTIFACTS_DIR)

for path in sorted(pathlib.Path(ARTIFACTS_DIR).glob("*.zip")):
    print("ZIP:", path)


## Optional: Download Artifact ZIP in Colab

```python
from google.colab import files
import pathlib
zips = sorted(pathlib.Path('/content/manovarta_outputs/artifacts').glob('*.zip'))
if zips:
    files.download(str(zips[-1]))
```
